In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("notebook-spark-cluster-ui-fixed")
    .master("spark://spark-master:7077")
    .config("spark.ui.showConsoleProgress", "true")
    .config("spark.sql.shuffle.partitions", "6")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.host", "host.docker.internal")
    .config("spark.driver.port", "7078")
    .config("spark.blockManager.port", "7079")
    .config("spark.ui.port", "4040")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("INFO")
print("Spark master UI: http://localhost:8080")
print("Worker UIs: http://localhost:8081 and http://localhost:8082")
print("Driver UI (this notebook session): http://localhost:4040")
spark

Spark master UI: http://localhost:8080
Worker UIs: http://localhost:8081 and http://localhost:8082
Driver UI (this notebook session): http://localhost:4040


In [ ]:
# Cluster-safe demo via JVM-side SQL
spark.sql("""
CREATE OR REPLACE TEMP VIEW notebook_orders AS
SELECT 1 AS id, CAST(10.5 AS DOUBLE) AS amount, 'USD' AS currency
UNION ALL
SELECT 2 AS id, CAST(25.0 AS DOUBLE) AS amount, 'EUR' AS currency
UNION ALL
SELECT 3 AS id, CAST(42.8 AS DOUBLE) AS amount, 'USD' AS currency
""")

agg_df = spark.sql("""
SELECT
  currency,
  count(*) AS rows_cnt,
  round(sum(amount), 2) AS amount_sum,
  round(avg(amount), 2) AS amount_avg
FROM notebook_orders
GROUP BY currency
ORDER BY currency
""")

agg_df.show()

In [ ]:
(
    agg_df.write
    .mode("overwrite")
    .format("parquet")
    .save("s3a://raw/notebook/spark_agg_demo")
)

read_back = spark.read.parquet("s3a://raw/notebook/spark_agg_demo")
read_back.show()